In [ ]:
# %pip install ninja ipykernel ipywidgets huggingface_hub --break-system-packages

In [ ]:
# %pip install --upgrade transformers --break-system-packages

In [ ]:
# %pip install git+https://github.com/intel/auto-round.git --break-system-packages

In [ ]:
# %pip install compressed-tensors --break-system-packages

In [ ]:
# %pip install git+https://github.com/sustcsonglin/flash-linear-attention.git --no-build-isolation --break-system-packages

In [ ]:
# %pip install git+https://github.com/Dao-AILab/causal-conv1d.git --no-build-isolation --break-system-packages

In [1]:
import os

import torch
from auto_round import AutoRound
from huggingface_hub import HfApi, create_repo, get_token, notebook_login
from safetensors import safe_open
from transformers import AutoModelForImageTextToText, AutoProcessor


In [2]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [3]:
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")


PyTorch Version: 2.12.0+cu130
CUDA Available: True
CUDA Version: 13.0
GPU Name: NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM: 95.0 GB


In [4]:
notebook_login()

In [5]:
MODEL_ID = "Qwen/Qwen3.8-27B"
HF_USER = "Vishva007"
OUTPUT_BASE_DIR = "./AutoRound"
LOCAL_PATH = "./local_model"

In [6]:
!hf download $MODEL_ID --local-dir $LOCAL_PATH

Hint: A new version of huggingface_hub (1.27.0) is available! You are using version 1.18.0.
To update, run: hf update
Fetching 32 files:   0%|                                | 0/32 [00:00<?, ?it/s]Still waiting to acquire lock on /workspace/local_model/.cache/huggingface/.gitignore.lock (elapsed: 0.1 seconds)
Still waiting to acquire lock on /workspace/local_model/.cache/huggingface/.gitignore.lock (elapsed: 0.1 seconds)
Still waiting to acquire lock on /workspace/local_model/.cache/huggingface/.gitignore.lock (elapsed: 0.1 seconds)
Still waiting to acquire lock on /workspace/local_model/.cache/huggingface/.gitignore.lock (elapsed: 0.1 seconds)
Fetching 32 files: 100%|███████████████████████| 32/32 [00:24<00:00,  1.33it/s]
Download complete: 100%|██████████████████| 55.6G/55.6G [00:24<00:00, 1.39GB/s]✓ Downloaded
  path: /workspace/local_model
Download complete: 100%|██████████████████| 55.6G/55.6G [00:24<00:00, 2.31GB/s]


In [7]:
for file in os.listdir(LOCAL_PATH):
    if file.endswith(".safetensors"):
        path = os.path.join(LOCAL_PATH, file)
        print(f"\nChecking {file}")

        with safe_open(path, framework="pt") as f:
            keys = list(f.keys())

            mtp_keys = [k for k in keys if "mtp" in k.lower()]
            for k in mtp_keys:
                print(k)


Checking model-00016-of-00018.safetensors

Checking model-00018-of-00018.safetensors
mtp.fc.weight
mtp.layers.0.input_layernorm.weight
mtp.layers.0.mlp.down_proj.weight
mtp.layers.0.mlp.gate_proj.weight
mtp.layers.0.mlp.up_proj.weight
mtp.layers.0.post_attention_layernorm.weight
mtp.layers.0.self_attn.k_norm.weight
mtp.layers.0.self_attn.k_proj.weight
mtp.layers.0.self_attn.o_proj.weight
mtp.layers.0.self_attn.q_norm.weight
mtp.layers.0.self_attn.q_proj.weight
mtp.layers.0.self_attn.v_proj.weight
mtp.norm.weight
mtp.pre_fc_norm_embedding.weight
mtp.pre_fc_norm_hidden.weight

Checking model-00014-of-00018.safetensors

Checking model-00012-of-00018.safetensors

Checking model-00017-of-00018.safetensors

Checking model-00013-of-00018.safetensors

Checking model-00015-of-00018.safetensors

Checking model-00010-of-00018.safetensors

Checking model-00011-of-00018.safetensors

Checking model-00001-of-00018.safetensors

Checking model-00009-of-00018.safetensors

Checking model-00006-of-00018.

In [8]:
model = AutoModelForImageTextToText.from_pretrained(
    LOCAL_PATH, 
    dtype=torch.bfloat16, 
    device_map="auto"
)
processor = AutoProcessor.from_pretrained(LOCAL_PATH)

tokenizer = processor.tokenizer


Loading weights:   0%|          | 0/1184 [00:00<?, ?it/s]

[ERROR] `min_frames` is part of Qwen3VLVideoProcessorInitKwargs, but not documented. Make sure to add it to the docstring of the function in /usr/local/lib/python3.12/dist-packages/transformers/models/qwen3_vl/video_processing_qwen3_vl.py.
[ERROR] `max_frames` is part of Qwen3VLVideoProcessorInitKwargs, but not documented. Make sure to add it to the docstring of the function in /usr/local/lib/python3.12/dist-packages/transformers/models/qwen3_vl/video_processing_qwen3_vl.py.


In [9]:
model

Qwen3_5ForConditionalGeneration(
  (model): Qwen3_5Model(
    (visual): Qwen3_5VisionModel(
      (patch_embed): Qwen3_5VisionPatchEmbed(
        (proj): Conv3d(3, 1152, kernel_size=(2, 16, 16), stride=(2, 16, 16))
      )
      (pos_embed): Embedding(2304, 1152)
      (rotary_pos_emb): Qwen3_5VisionRotaryEmbedding()
      (blocks): ModuleList(
        (0-26): 27 x Qwen3_5VisionBlock(
          (norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True, bias=True)
          (norm2): LayerNorm((1152,), eps=1e-06, elementwise_affine=True, bias=True)
          (attn): Qwen3_5VisionAttention(
            (qkv): Linear(in_features=1152, out_features=3456, bias=True)
            (proj): Linear(in_features=1152, out_features=1152, bias=True)
          )
          (mlp): Qwen3_5VisionMLP(
            (linear_fc1): Linear(in_features=1152, out_features=4304, bias=True)
            (linear_fc2): Linear(in_features=4304, out_features=1152, bias=True)
            (act_fn): GELUTanh()
         

In [11]:
def push_to_hub(local_dir, repo_name, token):
    """Creates repo and uploads folder to Hugging Face."""
    full_repo_id = f"{HF_USER}/{repo_name}"
    print(f"\n[Hub] Pushing {local_dir} to {full_repo_id}...")

    try:
        api = HfApi()
        create_repo(
            full_repo_id, repo_type="model", exist_ok=True, private=False, token=token
        )

        api.upload_folder(
            folder_path=local_dir, repo_id=full_repo_id, repo_type="model", token=token
        )
        print(f"[Hub] ✅ Successfully uploaded: https://huggingface.co/{full_repo_id}")
    except Exception as e:
        print(f"[Hub] ❌ Error uploading: {e}")

In [10]:
TUNING_CONFIG = {
    "group_size": 16,
    "sym": True,
    "iters": 400,  # High accuracy (Production grade)
    "nsamples": 512,  # More calibration data
    "batch_size": 4,  # Faster on 48GB VRAM
    "seqlen": 2048,
    "low_gpu_mem_usage": False,  # Keep on GPU for speed
    "enable_torch_compile": True,  # JIT acceleration
    "quant_nontext_module": False,  # Keep Vision Tower in FP16 (Crucial for VLM accuracy)
    "layer_config": {
        "mtp": {"data_type": "bfloat16"},
        "mtp.fc": {"data_type": "bfloat16"}
    }
}

In [12]:
ar = AutoRound(
    model=model,
    tokenizer=tokenizer,
    processor=processor,
    scheme="NVFP4",
    **TUNING_CONFIG,
)

2026-08-15 15:16:52 WARNING autoround.py L554: Passing 'group_size' directly to AutoRound is supported, but the recommended usage is 'alg_configs=SignRoundConfig(...)'.
2026-08-15 15:16:52 WARNING autoround.py L554: Passing 'sym' directly to AutoRound is supported, but the recommended usage is 'alg_configs=SignRoundConfig(...)'.
2026-08-15 15:16:52 WARNING autoround.py L554: Passing 'iters' directly to AutoRound is supported, but the recommended usage is 'alg_configs=SignRoundConfig(...)'.


In [ ]:
ar.quantize_and_save(
    OUTPUT_BASE_DIR, format="llm_compressor", inplace=True
)

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.
2026-08-15 15:16:54 WARNING logging.py L340: reset enable_torch_compile to `False` as activation is static
2026-08-15 15:16:55 INFO orchestrator.py L570: start to cache block inputs
2026-08-15 15:16:55 INFO mllm.py L86: Using MLLM template: qwen3_5
2026-08-15 15:16:55 INFO calib_dataset.py L1113: Preprocessing calibration dataset in a subprocess to avoid memory leaks...
2026-08-15 15:17:49 INFO device.py L1448: 'peak_ram': 62.48GB, 'peak_vram': 51.09GB
2026-08-15 15:17:49 INFO orchestrator.py L602: caching done
Quantizing model.language_model.layers.0:   0%|          | 0/64 [00:01<?, ?it/s]quantized 8/8 layers in the block, loss iter 0: 0.000027 -> iter 330: 0.000008
2026-08-15 15:21:37 INFO device.py L1448: 'peak_ram': 62.48GB, 'peak_vram': 53.37GB
Quantizing model.language_model.layers.1:   2%|▏         | 1/64 [03:48<3:59:31, 228.12s/it]quantized 8/8 layers in the 

In [ ]:
base_name = MODEL_ID.split("/")[-1]
hf_token = get_token()

In [ ]:
if hf_token:
    push_to_hub(
        os.path.join(OUTPUT_BASE_DIR, "local_model-w4g64/auto-round-auto-gptq"), 
        f"{base_name}-W4A16-AutoRound", 
        hf_token)
    push_to_hub(
        os.path.join(OUTPUT_BASE_DIR, "local_model-w4g64/auto-gptq"), 
        f"{base_name}-W4A16-AutoRound-GPTQ",
        hf_token
    )
else:
    print("No Hugging Face token found. Skipping upload to hub.")